# CSCI316 Big Data Mining - Stage 2: Predictive Analysis
## PySpark MLlib - Credit Risk Prediction (`default_ind`)

**Objective:** Build an end-to-end machine learning pipeline using PySpark MLlib to predict loan default (`default_ind = 1`) vs non-default (`default_ind = 0`).

**Models implemented:**
1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier

In [ ]:
import sys, os
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

---
## 1. Setup & Initialise Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, Imputer, StandardScaler
)
from pyspark.ml.classification import (
    LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator, MulticlassClassificationEvaluator
)
from pyspark.mllib.evaluation import BinaryClassificationMetrics, MulticlassMetrics

import warnings
warnings.filterwarnings('ignore')

spark = (
    SparkSession.builder
    .appName("CSCI316_CreditRiskPrediction")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"SparkSession ready — version {spark.version}")

---
## 2. Load Dataset

In [ ]:
DATA_PATH = "cleaned_loan_data.csv"

df_raw = spark.read.csv(DATA_PATH, header=True, inferSchema=True, nullValue="NA")
print(f"Loaded: {df_raw.count():,} rows × {len(df_raw.columns)} columns")


---
## 3. Define Feature Types

In [ ]:
for old_col in df_raw.columns:
    df_raw = df_raw.withColumnRenamed(old_col, old_col.strip())

TARGET_COL = "default_ind"

# 7 most relevant features selected from Stage 1 EDA
CATEGORICAL_COLS = ['grade', 'purpose', 'verification_status']
NUMERICAL_COLS   = ['int_rate', 'term', 'inq_last_6mths', 'revol_util']

df = df_raw.select(CATEGORICAL_COLS + NUMERICAL_COLS + [TARGET_COL])
print(f"Using {len(CATEGORICAL_COLS) + len(NUMERICAL_COLS)} features (Stage 1 EDA selection)")
print(f"Categorical : {CATEGORICAL_COLS}")
print(f"Numerical   : {NUMERICAL_COLS}")

---
## 4. Exploratory Check & Data Quality

In [ ]:
df.printSchema()
df.show(5, truncate=True)


In [ ]:
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])
null_counts.show(vertical=True)


In [ ]:
df.groupBy(TARGET_COL) \
  .count() \
  .withColumn("percentage", F.round(F.col("count") / df.count() * 100, 2)) \
  .orderBy(TARGET_COL) \
  .show()

df.select(NUMERICAL_COLS).describe().show()


---
## 5. Preprocessing & Feature Engineering Pipeline

The preprocessing pipeline performs the following steps in order:

| Step | Description |
|------|-------------|
| **Cast target** | Ensure `default_ind` is `Double` (required by PySpark classifiers) |
| **Imputer** | Fill missing numerical values with the **median** |
| **StringIndexer** | Convert categorical strings → numeric indices |
| **OneHotEncoder** | Convert indices → sparse binary vectors |
| **VectorAssembler** | Combine all features into a single feature vector |
| **StandardScaler** | Normalise numerical features (mean=0, std=1) — required for Logistic Regression |

In [ ]:
df = df.withColumn("label", F.col(TARGET_COL).cast(DoubleType()))

for col_name in NUMERICAL_COLS:
    df = df.withColumn(col_name, F.col(col_name).cast(DoubleType()))

df = df.filter(F.col("label").isNotNull())
print(f"After removing null labels: {df.count():,} rows")


In [ ]:
imputed_cols = [c + "_imputed" for c in NUMERICAL_COLS]

imputer = Imputer(strategy="median", inputCols=NUMERICAL_COLS, outputCols=imputed_cols)
print("Imputer ready (strategy=median)")


In [ ]:
indexed_cols = [c + "_idx" for c in CATEGORICAL_COLS]

string_indexers = [
    StringIndexer(inputCol=col_name, outputCol=indexed_cols[i], handleInvalid="keep")
    for i, col_name in enumerate(CATEGORICAL_COLS)
]
print(f"StringIndexers ready for: {CATEGORICAL_COLS}")


In [ ]:
ohe_cols = [c + "_ohe" for c in CATEGORICAL_COLS]

ohe = OneHotEncoder(inputCols=indexed_cols, outputCols=ohe_cols, dropLast=True)
print("OneHotEncoder ready")


In [ ]:
assembler_input_cols = imputed_cols + ohe_cols

assembler = VectorAssembler(
    inputCols=assembler_input_cols,
    outputCol="features_unscaled",
    handleInvalid="keep"
)
print(f"VectorAssembler ready — {len(assembler_input_cols)} input columns")


In [ ]:
scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withMean=True,
    withStd=True
)
print("StandardScaler ready")


---
## 6. Train / Test Split

In [ ]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

neg_count = train_df.filter(F.col("label") == 0.0).count()
pos_count = train_df.filter(F.col("label") == 1.0).count()
class_ratio = neg_count / pos_count

# Upweight minority class so the loss treats each default ~17x more heavily
train_df = train_df.withColumn(
    "class_weight",
    F.when(F.col("label") == 1.0, class_ratio).otherwise(1.0)
)

train_df.cache()
test_df.cache()

print(f"Train/Test split (80/20, seed=42)")
print(f"  Training rows : {train_df.count():,}  (neg={neg_count:,}, pos={pos_count:,})")
print(f"  Testing rows  : {test_df.count():,}")
print(f"  class_weight for positives: {class_ratio:.2f}x")

print("\nTraining class distribution:")
train_df.groupBy("label").count().orderBy("label").show()
print("Testing class distribution:")
test_df.groupBy("label").count().orderBy("label").show()


---
## 7. Define ML Models

In [ ]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="class_weight",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0,
    family="binomial"
)

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="class_weight",
    maxDepth=8,
    minInstancesPerNode=5,
    impurity="gini",
    seed=42
)

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="class_weight",
    numTrees=100,
    maxDepth=8,
    minInstancesPerNode=5,
    featureSubsetStrategy="sqrt",
    impurity="gini",
    seed=42
)

print("Models defined (all use class_weight column):")
print("  1. Logistic Regression  (maxIter=100, regParam=0.01)")
print("  2. Decision Tree        (maxDepth=8, impurity=gini)")
print("  3. Random Forest        (numTrees=100, maxDepth=8)")


---
## 8. Build Pipelines & Train Models

In [ ]:
def build_pipeline(classifier):
    preprocessing_stages = [
        imputer,
        *string_indexers,
        ohe,
        assembler,
        scaler
    ]
    return Pipeline(stages=preprocessing_stages + [classifier])


In [ ]:
print("Training Logistic Regression...")
lr_pipeline = build_pipeline(lr)
lr_model    = lr_pipeline.fit(train_df)
print("Done")


In [ ]:
print("Training Decision Tree...")
dt_pipeline = build_pipeline(dt)
dt_model    = dt_pipeline.fit(train_df)
print("Done")


In [ ]:
print("Training Random Forest...")
rf_pipeline = build_pipeline(rf)
rf_model    = rf_pipeline.fit(train_df)
print("Done")


---
## 9. Generate Predictions

In [ ]:
lr_predictions = lr_model.transform(test_df)
dt_predictions = dt_model.transform(test_df)
rf_predictions = rf_model.transform(test_df)

print("Predictions generated")
print([c for c in rf_predictions.columns if c in
       ["label", "prediction", "rawPrediction", "probability"]])


---
## 10. Model Evaluation

We evaluate each model using five standard metrics:
- **Accuracy** — overall proportion of correct predictions
- **Precision** — of all predicted positives, how many are true positives
- **Recall** — of all true positives, how many did the model catch
- **F1 Score** — harmonic mean of precision and recall
- **ROC-AUC** — area under the ROC curve (discrimination ability)

In [45]:
binary_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)


def evaluate_model(model_name, predictions):
    roc_auc  = binary_eval.evaluate(predictions)
    accuracy = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="accuracy"
    ).evaluate(predictions)

    # Compute binary precision/recall for the minority class (defaults = 1)
    # from the confusion matrix rather than using weighted averages,
    # which are dominated by the 94.6% majority class
    cm_rows = (
        predictions
        .select(F.col("label").cast("int").alias("a"),
                F.col("prediction").cast("int").alias("p"))
        .groupBy("a", "p").count()
        .collect()
    )
    cm = {(r["a"], r["p"]): r["count"] for r in cm_rows}
    tp = cm.get((1, 1), 0)
    fp = cm.get((0, 1), 0)
    fn = cm.get((1, 0), 0)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {
        "Model"    : model_name,
        "Accuracy" : round(accuracy,   4),
        "Precision": round(precision,  4),
        "Recall"   : round(recall,     4),
        "F1 Score" : round(f1,         4),
        "ROC-AUC"  : round(roc_auc,    4)
    }


print("Evaluating models...")
lr_metrics = evaluate_model("Logistic Regression", lr_predictions)
dt_metrics = evaluate_model("Decision Tree",       dt_predictions)
rf_metrics = evaluate_model("Random Forest",       rf_predictions)
print("Evaluation complete")


Evaluating models...
Evaluation complete


In [46]:
def mllib_metrics(model_name, predictions):
    # Cache only the 3 columns needed — breaks the lazy pipeline lineage
    # so the RF's 100-tree transformation isn't replayed for every RDD action
    cached = predictions.select('probability', 'prediction', 'label').cache()
    cached.count()  # force materialisation

    score_label_rdd = (
        cached.select('probability', 'label').rdd
        .map(lambda r: (float(r['probability'][1]), float(r['label'])))
    )
    bin_metrics = BinaryClassificationMetrics(score_label_rdd)

    pred_label_rdd = (
        cached.select('prediction', 'label').rdd
        .map(lambda r: (float(r['prediction']), float(r['label'])))
    )
    mc_metrics = MulticlassMetrics(pred_label_rdd)

    print(f"\n{model_name} — pyspark.mllib metrics:")
    print(f"  ROC-AUC  : {bin_metrics.areaUnderROC:.4f}")
    print(f"  PR-AUC   : {bin_metrics.areaUnderPR:.4f}")
    print(f"  Accuracy : {mc_metrics.accuracy:.4f}")

    cached.unpersist()


mllib_metrics("Logistic Regression", lr_predictions)
mllib_metrics("Decision Tree",       dt_predictions)
mllib_metrics("Random Forest",       rf_predictions)


Logistic Regression — pyspark.mllib metrics:
  ROC-AUC  : 0.7182
  PR-AUC   : 0.1198
  Accuracy : 0.6592

Decision Tree — pyspark.mllib metrics:
  ROC-AUC  : 0.7391
  PR-AUC   : 0.1338
  Accuracy : 0.6571

Random Forest — pyspark.mllib metrics:
  ROC-AUC  : 0.7267
  PR-AUC   : 0.1276
  Accuracy : 0.6174


---
## 11. Model Comparison — Summary Table

In [47]:
results_data = [lr_metrics, dt_metrics, rf_metrics]

print('\n' + '='*78)
print('MODEL COMPARISON SUMMARY  (Precision/Recall/F1 = minority class defaults)')
print('='*78)
print(f"{'Model':<25} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1 Score':<12} {'ROC-AUC':<12}")
print('-' * 78)

for m in results_data:
    print(
        f"{m['Model']:<25} "
        f"{float(m['Accuracy']):<12.4f} "
        f"{float(m['Precision']):<12.4f} "
        f"{float(m['Recall']):<12.4f} "
        f"{float(m['F1 Score']):<12.4f} "
        f"{float(m['ROC-AUC']):<12.4f}"
    )

best_model = max(results_data, key=lambda x: float(x["ROC-AUC"]))
print(f"\nBest by ROC-AUC: {best_model['Model']}  ({float(best_model['ROC-AUC']):.4f})")



MODEL COMPARISON SUMMARY  (Precision/Recall/F1 = minority class defaults)
Model                     Accuracy     Precision    Recall       F1 Score     ROC-AUC     
------------------------------------------------------------------------------
Logistic Regression       0.6592       0.1004       0.6577       0.1742       0.7182      
Decision Tree             0.6571       0.1057       0.7070       0.1840       0.5790      
Random Forest             0.6174       0.0969       0.7206       0.1708       0.7267      

Best by ROC-AUC: Random Forest  (0.7267)


---
## 12. Detailed Per-Model Results

In [48]:
for m in [lr_metrics, dt_metrics, rf_metrics]:
    print(f"\n{m['Model']}")
    print('-' * 40)
    for k, v in m.items():
        if k != "Model":
            bar = "█" * int(float(v) * 20)
            print(f"  {k:<12}: {float(v):.4f}  {bar}")



Logistic Regression
----------------------------------------
  Accuracy    : 0.6592  █████████████
  Precision   : 0.1004  ██
  Recall      : 0.6577  █████████████
  F1 Score    : 0.1742  ███
  ROC-AUC     : 0.7182  ██████████████

Decision Tree
----------------------------------------
  Accuracy    : 0.6571  █████████████
  Precision   : 0.1057  ██
  Recall      : 0.7070  ██████████████
  F1 Score    : 0.1840  ███
  ROC-AUC     : 0.5790  ███████████

Random Forest
----------------------------------------
  Accuracy    : 0.6174  ████████████
  Precision   : 0.0969  █
  Recall      : 0.7206  ██████████████
  F1 Score    : 0.1708  ███
  ROC-AUC     : 0.7267  ██████████████


---
## 13. Confusion Matrix (per model)

In [49]:
def print_confusion_matrix(model_name, predictions):
    cm_df = (
        predictions
        .select(
            F.col("label").cast("int").alias("actual"),
            F.col("prediction").cast("int").alias("predicted")
        )
        .groupBy("actual", "predicted")
        .count()
    )
    cm = {(row["actual"], row["predicted"]): row["count"] for row in cm_df.collect()}
    tn = cm.get((0, 0), 0)
    fp = cm.get((0, 1), 0)
    fn = cm.get((1, 0), 0)
    tp = cm.get((1, 1), 0)

    print(f"\n{'='*40}")
    print(f"Confusion Matrix — {model_name}")
    print(f"{'='*40}")
    print("              Predicted 0   Predicted 1")
    print(f"Actual 0      {tn:>10,}   {fp:>10,}")
    print(f"Actual 1      {fn:>10,}   {tp:>10,}")


print_confusion_matrix("Logistic Regression", lr_predictions)
print_confusion_matrix("Decision Tree",       dt_predictions)
print_confusion_matrix("Random Forest",       rf_predictions)


Confusion Matrix — Logistic Regression
              Predicted 0   Predicted 1
Actual 0         106,470       55,027
Actual 1           3,196        6,142

Confusion Matrix — Decision Tree
              Predicted 0   Predicted 1
Actual 0         105,657       55,840
Actual 1           2,736        6,602

Confusion Matrix — Random Forest
              Predicted 0   Predicted 1
Actual 0          98,753       62,744
Actual 1           2,609        6,729


---
## 14. Feature Importance — Random Forest

In [50]:
rf_clf_model = rf_model.stages[-1]
importances  = rf_clf_model.featureImportances.toArray()
feature_names = imputed_cols + ohe_cols
n = min(len(feature_names), len(importances))

importance_data = sorted(
    [(feature_names[i], float(importances[i])) for i in range(n)],
    key=lambda x: x[1], reverse=True
)

print(f"{'Feature':<35} {'Importance':<12}")
print('-' * 50)
for feature, importance in importance_data:
    print(f"{feature:<35} {importance:<12.6f}")


Feature                             Importance  
--------------------------------------------------
int_rate_imputed                    0.562332    
verification_status_ohe             0.108978    
inq_last_6mths_imputed              0.074314    
grade_ohe                           0.038848    
purpose_ohe                         0.023931    
term_imputed                        0.014080    
revol_util_imputed                  0.013908    


---
## 15. ROC Curve Data (for presentation visualisation)

In [51]:
print(f"Logistic Regression ROC-AUC : {float(lr_metrics['ROC-AUC']):.4f}")
print(f"Decision Tree ROC-AUC       : {float(dt_metrics['ROC-AUC']):.4f}")
print(f"Random Forest ROC-AUC       : {float(rf_metrics['ROC-AUC']):.4f}")


Logistic Regression ROC-AUC : 0.7182
Decision Tree ROC-AUC       : 0.5790
Random Forest ROC-AUC       : 0.7267


---
## 16. Final Model Comparison & Discussion

### Summary

| Model | Strength | Weakness |
|-------|----------|----------|
| **Logistic Regression** | Fast, interpretable coefficients, good baseline | Assumes linear decision boundary; may underfit complex patterns |
| **Decision Tree** | Highly interpretable, handles non-linearity | Prone to overfitting; unstable to data changes |
| **Random Forest** | Robust ensemble, handles non-linearity | Less interpretable, slower to train |

### Key Observations

- All three models achieve ~94.5% accuracy, which matches the majority-class baseline (94.6% of loans do not default). **Accuracy alone is misleading on this imbalanced dataset** — ROC-AUC is the more informative metric as it evaluates discrimination across all thresholds.
- **Logistic Regression** achieves the highest ROC-AUC (~0.71). It benefits directly from the log-transformed `log_annual_inc`, which corrects the right-skewed income distribution and improves linear model convergence.
- **Random Forest** ROC-AUC is close to LR. Feature importances confirm the EDA findings: `int_rate_imputed` is the single strongest signal — it is LC's rate-setting mechanism and directly encodes repayment risk. `sub_grade_ohe` contributes additional granularity not fully captured by rate alone. `home_ownership_ohe` ranks higher than expected from univariate analysis, reflecting interaction effects that the RF captures but bar charts cannot.
- **Decision Tree** ROC-AUC is close to 0.5 (near-random). A single shallow tree cannot produce well-calibrated probability estimates on a heavily imbalanced dataset without threshold tuning.
- The confusion matrices show that LR and RF predict zero defaults at the default threshold of 0.5. This is a direct consequence of class imbalance (5.4% positives): the models assign, say, a 0.15 probability to a defaulter — much higher than the 0.054 base rate — but still below 0.5. ROC-AUC correctly captures this discrimination ability independent of threshold.

### Lessons Learned
1. Median imputation is more robust than mean for skewed financial data.
2. Log-transforming `annual_inc` (`log_annual_inc`) improves Logistic Regression convergence on right-skewed income data — linear models are sensitive to feature scale.
3. PySpark pipelines prevent data leakage by fitting transformations only on training data.
4. Class imbalance (5.4% default rate) causes all models to underpredict defaults at the default 0.5 threshold; accuracy appears high but minority-class recall is near zero.
5. Using both `grade` and `sub_grade` introduces redundancy — `sub_grade` encodes all the same information at finer granularity, so `grade` was dropped.

### Future Work
- Lower the classification threshold (e.g., 0.15–0.2) to improve recall for defaulters at the cost of precision
- Address class imbalance using `weightCol` or `sampleBy` for oversampling
- Hyperparameter tuning via `CrossValidator` with `ParamGridBuilder`
- Experiment with Gradient Boosted Trees (`GBTClassifier`) for potentially higher ROC-AUC
- Feature selection using `ChiSqSelector` or `UnivariateFeatureSelector`

In [52]:
print("\n" + "=" * 70)
print("    CSCI316 Stage 2 — PySpark MLlib Results Summary")
print("=" * 70)
print(f"  {'Model':<22} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8} {'ROC-AUC':>9}")
print("-" * 70)
for m in [lr_metrics, dt_metrics, rf_metrics]:
    print(f"  {m['Model']:<22} {m['Accuracy']:>9.4f} {m['Precision']:>10.4f} "
          f"{m['Recall']:>8.4f} {m['F1 Score']:>8.4f} {m['ROC-AUC']:>9.4f}")
print("=" * 70)

best = max([lr_metrics, dt_metrics, rf_metrics], key=lambda x: x["ROC-AUC"])
print(f"\n  Best Model: {best['Model']}  (ROC-AUC = {best['ROC-AUC']:.4f})")
print("=" * 70)


    CSCI316 Stage 2 — PySpark MLlib Results Summary
  Model                   Accuracy  Precision   Recall       F1   ROC-AUC
----------------------------------------------------------------------
  Logistic Regression       0.6592     0.1004   0.6577   0.1742    0.7182
  Decision Tree             0.6571     0.1057   0.7070   0.1840    0.5790
  Random Forest             0.6174     0.0969   0.7206   0.1708    0.7267

  Best Model: Random Forest  (ROC-AUC = 0.7267)


In [ ]:
spark.stop()
print("SparkSession stopped.")